# SBi-FAGAT Tier 2: Macro Signed User-Interaction Social Network & Temporal Link Prediction (T4 GPU)

Runs the **Two-Tier** macro polarization and chronological signed link-prediction suite across **VN** and **PHEME**:
1. **Macro Structural Polarization**: Structural Balance Theory (SBT triangle ratio + degree-preserving permutation $z$-score), Signed Laplacian bipartition, Signed Modularity $Q_{\text{signed}}$, and Garimella Random Walk Controversy (RWC).
2. **Temporal Signed Link Prediction (6 models x 5 seeds)**: `UserTextMLP`, `StandardGCN`, `StandardGATv2`, `SGCN`, `FAGCN`, and `SBiFAGAT-Link` with strict chronological train-only features and intra-batch target-pair masking.
3. **Artifact Assembly**: Merges Tier 1 + Tier 2 outputs via `scripts.assemble_paper_artifacts` into `sbifagat_macro_signed_bundle.zip`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content") if Path("/content").exists() else Path.cwd()
REPO_URL = os.environ.get("SBIF_REPO_URL", "https://github.com/hieu-kiien/gnn-nlp-propagation.git")
BRANCH = os.environ.get("SBIF_BRANCH", "results/kaggle-v5-vn-v3-temporal-ablation")

if (Path.cwd() / "scripts" / "run_macro_signed_network.py").exists():
    REPO_DIR = Path.cwd()
else:
    REPO_DIR = WORK_ROOT / "gnn-nlp-propagation"
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

print("REPO_DIR:", REPO_DIR)
subprocess.run(["nvidia-smi"], check=False)

# Preserve existing CUDA PyTorch build while ensuring lightweight dependencies
req_pkgs = ["pyarrow", "pydantic", "scikit-learn", "scipy", "matplotlib", "tqdm"]
try:
    import torch_geometric  # noqa: F401
except ImportError:
    req_pkgs.append("torch-geometric")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *req_pkgs])

import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# Fast verification gate before GPU experiments
subprocess.check_call([sys.executable, "-u", "-m", "pytest", "-q", "tests/test_macro_signed_network.py", "tests/test_rumoureval_sdqc.py"])

REQUIRED_FILES = ("nodes.parquet", "edges.parquet", "nlp_features.pt")


def _is_valid_processed_dir(path: Path) -> bool:
    return path.is_dir() and all((path / f).exists() for f in REQUIRED_FILES)


def _discover_processed_corpora() -> dict[str, Path]:
    found: dict[str, Path] = {}
    mounts = [Path("data/processed"), Path("/kaggle/input"), Path("/kaggle/working"), Path("/content/drive/MyDrive")]
    for ds_name in ("vn", "pheme"):
        direct = Path("data/processed") / ds_name
        if _is_valid_processed_dir(direct):
            found[ds_name] = direct
            continue
        for root in mounts:
            if not root.exists():
                continue
            for candidate in root.rglob("nodes.parquet"):
                cdir = candidate.parent
                if not _is_valid_processed_dir(cdir):
                    continue
                hint = "/".join(cdir.parts).lower()
                if ds_name == "vn" and ("vn" in hint or "vietnam" in hint):
                    found["vn"] = cdir
                    break
                if ds_name == "pheme" and "pheme" in hint:
                    found["pheme"] = cdir
                    break
            if ds_name in found:
                break
    return found


CORPORA = _discover_processed_corpora()
print("Discovered corpora:", {k: str(v) for k, v in CORPORA.items()})

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

EPOCHS = int(os.environ.get("MACRO_EPOCHS", "40"))
PATIENCE = int(os.environ.get("MACRO_PATIENCE", "8"))
BATCH_SIZE = int(os.environ.get("MACRO_BATCH_SIZE", "2048"))
HIDDEN_DIM = int(os.environ.get("MACRO_HIDDEN_DIM", "64"))

if not CORPORA:
    raise RuntimeError(
        "No processed VN or PHEME dataset (nodes.parquet, edges.parquet, nlp_features.pt) found. "
        "Attach the processed Kaggle Dataset or Member1/Member2 bundle before running."
    )

for ds_name in ("vn", "pheme"):
    ds_dir = CORPORA.get(ds_name)
    if ds_dir is None:
        print(f"Skipping {ds_name}: not attached in this session.")
        continue
    print(f"\n=== Running Tier 2 Macro Network + 5-Seed Signed Link GNN Benchmark ({ds_name}) ===", flush=True)
    subprocess.check_call(
        [
            sys.executable,
            "-u",
            "-m",
            "scripts.run_macro_signed_network",
            "--data-dir",
            str(ds_dir),
            "--dataset-name",
            ds_name,
            "--epochs",
            str(EPOCHS),
            "--patience",
            str(PATIENCE),
            "--batch-size",
            str(BATCH_SIZE),
            "--hidden-dim",
            str(HIDDEN_DIM),
            "--paper-dir",
            "paper",
        ]
    )

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

run_rumour = os.environ.get("RUN_RUMOUREVAL", "auto").lower()
rumour_candidates = (
    Path("paper/results_rumoureval_sdqc.json"),
    Path("paper/results_rumoureval_sdqc_en.json"),
)
rumour_json = Path("paper/results_rumoureval_sdqc.json")
rumour_has_root = False
for p in rumour_candidates:
    if p.exists():
        try:
            rumour_has_root = json.loads(p.read_text(encoding="utf-8")).get("meta", {}).get("context_mode") == "root"
        except Exception:
            rumour_has_root = False
        if rumour_has_root:
            break
if run_rumour == "1" or (run_rumour == "auto" and not rumour_has_root):
    print("=== Running RumourEval-2019 SDQC Root-Context NLI Verification ===", flush=True)
    subprocess.check_call(
        [
            sys.executable,
            "-u",
            "-m",
            "scripts.eval_rumoureval_sdqc",
            "--context-mode",
            "root",
            "--output-json",
            str(rumour_json),
        ]
    )
else:
    print("Skipping RumourEval-2019 check (already present or RUN_RUMOUREVAL=0).")

In [ ]:
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.check_call([sys.executable, "-u", "-m", "scripts.assemble_paper_artifacts", "--paper-dir", "paper"])
manifest = json.loads(Path("paper/assembly_manifest.json").read_text(encoding="utf-8"))
print("Assembly underscore issues:", manifest.get("latex_underscore_issues"))

out_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
bundle_zip = out_root / "sbifagat_macro_signed_bundle.zip"
FORBIDDEN_SUBSTR = (".secret", ".env", "_machine_key.csv")
ALLOWED_EXT = {".json", ".tex", ".pdf", ".svg", ".md"}

with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(Path("paper").rglob("*")):
        if not fpath.is_file() or fpath.suffix.lower() not in ALLOWED_EXT:
            continue
        low = fpath.name.lower()
        if any(tok in low for tok in FORBIDDEN_SUBSTR):
            continue
        zf.write(fpath, arcname=str(fpath))

print(f"Bundle written: {bundle_zip} ({bundle_zip.stat().st_size / 1024.0:.1f} KB)")